# Notebook 5 — The real-hardware run (set D), limitations, and what comes next

**What this notebook covers**

1. What is still missing, and why it is the part that matters most
2. What you need
3. The four steps, with the exact commands
4. What the circuit looks like on a real chip
5. The controls that are built in
6. A rehearsal of the analysis (simulated stand-in data)
7. How to read the real result when you have it
8. How many events do you need?
9. Every limitation of this project in one place
10. Ideas for going further

Code: `src/qon/hardware.py`, `src/qon/hardware_analysis.py`, `scripts/06_hardware_run.py`, `scripts/07_analyse_hardware.py`.

In [1]:
# Set-up: make the project's code importable and keep the output tidy.
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_rows", 60); pd.set_option("display.float_format", lambda v: f"{v:.4f}")
%matplotlib inline

## 1. What is still missing

Everything in Notebooks 3 and 4 was computed on a laptop. That covers five of the six feature sets. The sixth, **set D, the circuit run on a real IBM quantum computer, has not been run.** It could not be: it needs your IBM Quantum account and real quantum time.

It is also the part that matters most. HSBC's claim is specifically that *real hardware* helped where perfect simulation did not. Notebook 3 shows that everything we can simulate agrees with HSBC's own simulations (no gain). Only a hardware run can test the claim itself.

## 2. What you need

* An IBM Quantum account. The free **Open Plan** gives 10 minutes of quantum time per 28 days.
* Your API key saved once on this computer. Do this yourself in a Python prompt; never paste the key into a script, a notebook or a chat:

```python
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(token="<your key>", instance="<your instance CRN>")
```

## 3. The four steps

Run these from the project folder, in this order.

**Step 1 — dry run on a simulated chip (free, no account, about 2 minutes)**

```bash
.venv/bin/python scripts/06_hardware_run.py --mode local
```

Checks that the 109-qubit circuit fits on the chip, then runs a 12-qubit version of exactly the same pipeline on a simulated copy of ibm_fez and compares the answers with the exact simulation. (Simulating a *noisy* 109-qubit chip gate by gate would take hours, which is why the rehearsal circuit is short.)

**Step 2 — timing job on the real chip (a few seconds of quantum time)**

```bash
.venv/bin/python scripts/06_hardware_run.py --mode timing --backend ibm_fez
```

Runs 5 circuits. Look up the job's *usage* on quantum.cloud.ibm.com and divide by the number of circuits: that is your cost per circuit, and it tells you how many events your budget allows.

**Step 3 — the real run**

```bash
.venv/bin/python scripts/06_hardware_run.py --mode run --backend ibm_fez --events 400 --shots 1024
```

**Step 4 — analyse it**

```bash
.venv/bin/python scripts/07_analyse_hardware.py
```

Here is the output of the dry run (step 1) on this machine:

In [2]:
from qon import config as C
log = (C.ROOT / "results" / "logs" / "hardware_dry_run.log").read_text().splitlines()
print("\n".join(line for line in log if "arn" not in line))

DRY RUN on a simulated copy of ibm_fez (no account, no quantum time).
  full circuit fits: 109 qubits in a row, depth 80, 648 entangling gates
  executing the 12-qubit version on the simulated chip ...
backend: fake_fez   12-qubit chain starts [140, 141, 142, 143, 144, 145, 146, 147] ...
  median two-qubit gate error on the chain: 0.0031   worst: 0.0040
  transpiled circuit: depth 80, gates {'rz': 376, 'sx': 200, 'cz': 66, 'barrier': 4}
  plan: 12 events + 2 sentinel repeats = 14 circuits, 256 shots x 3 measurement bases each
  job 1: circuits 0..13 done
saved hardware_features_LOCAL_DRY_RUN.parquet
sentinel check (identical circuits repeated through the run):
{
  "n_sentinels": 2,
  "median_observed_std": 0.03104628284320904,
  "median_shot_noise_std": 0.05879874948271661,
  "excess_ratio": 0.5280092368688016,
  "median_shift_first_vs_second_half": 0.04390607425813736
}
agreement with the perfect-chip simulation: correlation 0.989, average shrink factor 0.949


**How to read it.** The last line is the important one: the values measured on the simulated chip agree with the exact simulation (correlation about 0.99), shrunk by a few percent. That is exactly what noise should do, and it confirms that the pipeline assigns each measurement to the right qubit.

## 4. What the circuit looks like on a real chip

IBM chips are not a straight line of qubits; they are a honeycomb ("heavy-hex"). Our circuit needs 109 qubits in a row, so `find_chain` looks for a snake-like path through the honeycomb, avoiding the worst couplers. The circuit is then translated ("transpiled") into the chip's own gates.

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeFez
from qon.circuit import make_spec
from qon.hardware import find_chain, prepare, chip_snapshot
fake = FakeFez()
spec = make_spec(n_features=98)
chain = find_chain(fake, spec.n_qubits)
prepared = prepare(spec, fake, chain)
snap = chip_snapshot(fake, chain)
ops = prepared.isa_circuit.count_ops()
print(f"chip: {fake.name} ({fake.num_qubits} qubits);  path of {len(chain)} connected qubits found")
print(f"path starts: {chain[:12]} ...")
print(f"two-qubit gates on the chip: {ops.get('cz', 0)}  (= 216 Heisenberg gates x 3)")
print(f"circuit depth after translation: {prepared.isa_circuit.depth()} gate layers")
print(f"typical two-qubit gate error on the path: {np.median(snap['two_qubit_error']):.2%}; worst: {max(snap['two_qubit_error']):.2%}")

chip: fake_fez (156 qubits);  path of 109 connected qubits found
path starts: [139, 135, 134, 133, 132, 131, 130, 129, 118, 109, 110, 111] ...
two-qubit gates on the chip: 648  (= 216 Heisenberg gates x 3)
circuit depth after translation: 80 gate layers
typical two-qubit gate error on the path: 0.39%; worst: 1.57%


A detail that matters for noise: written naively, each Heisenberg gate would need **six** entangling gates on the chip. We use a standard construction that needs **three** (`heisenberg_gate` in `circuit.py`; a test proves both are the same operation). Entangling gates are the noisiest thing a chip does, so this halves the damage.

## 5. The controls that are built in

These are the controls HSBC's critics asked for. They are fixed in [PREREGISTRATION.md](../PREREGISTRATION.md) before any quantum time is spent.

| Control | What it does | Why |
|---|---|---|
| **Random run order** | events are sent in a shuffled order, not by date | so that slow drift of the chip cannot act as a hidden clock (Notebook 2, section 9) |
| **Sentinel circuits** | one fixed circuit is repeated after every 20 events | identical inputs should give identical outputs; any extra scatter is drift, measured directly |
| **Same error handling as HSBC** | Pauli twirling + TREX, nothing else | so that the comparison is like for like |
| **Classical noise twin (set E)** | the perfect values, shrunk and jittered to match the hardware | to test whether hardware noise is anything more than classical noise |
| **Everything written down first** | one circuit, one seed, one protocol | so that nobody can pick the best of several attempts afterwards |

## 6. A rehearsal of the analysis

To make sure the analysis works before spending quantum time, `scripts/07_analyse_hardware.py --rehearsal` builds a **stand-in** results file from the noisy simulator (600 events, random order, with sentinels) and runs the complete analysis on it.

**These are simulated numbers. They say nothing about real hardware.** They only show what the output will look like.

In [4]:
import json
tab = pd.read_csv(C.TABLES / "partD_REHEARSAL_auc.csv")
summ = json.loads((C.TABLES / "partD_REHEARSAL_summary.json").read_text())
print(f"REHEARSAL (simulated): {summ['n_events']} events, {summ['n_test_events']} of them used for testing\n")
wide = tab.pivot(index="set", columns="model", values="auc")[["All", "LR", "RF", "XGB", "NN"]]
low = tab[tab.model == "All"].set_index("set")["ci_low"]; high = tab[tab.model == "All"].set_index("set")["ci_high"]
wide["All: 95% low"], wide["All: 95% high"] = low, high
wide.round(3)

REHEARSAL (simulated): 600 events, 294 of them used for testing



model,All,LR,RF,XGB,NN,All: 95% low,All: 95% high
set,,,,,,,
A,0.4950,0.4810,0.4970,0.4910,0.5160,0.4220,0.5700
B,0.4930,0.5060,0.5030,0.4930,0.4790,0.4270,0.5640
C,0.5100,0.5130,0.5360,0.5380,0.4630,0.4460,0.5700
D,0.4780,0.4960,0.5190,0.4560,0.4680,0.4120,0.5480
E,0.4930,0.5100,0.4980,0.4890,0.4710,0.4280,0.5620
F,0.4850,0.5090,0.4450,0.5040,0.4470,0.4340,0.5430


In [5]:
print("How the (simulated) chip distorts the perfect values:")
for k, v in summ["noise"].items(): print(f"   {k:32s} {v:.4f}")
print("\nDrift checks:")
for k, v in summ["drift"].items(): print(f"   {k:36s} {v:.4f}" if isinstance(v, float) else f"   {k:36s} {v}")

How the (simulated) chip distorts the perfect values:
   median_shrink                    0.8990
   median_jitter                    0.0166
   shot_noise_alone                 0.0151
   correlation_with_perfect_chip    0.9984

Drift checks:
   n_sentinels                          31
   median_observed_std                  0.0139
   median_shot_noise_std                0.0146
   excess_ratio                         0.9526
   median_shift_first_vs_second_half    0.0035
   correlation_run_order_vs_date        0.0264
   noise_reveals_run_position_auc       0.4733


**How to read the drift checks**

* `excess_ratio`: scatter of the repeated sentinel circuits divided by the scatter that shot noise alone would give. About 1 means no visible drift; well above 1 means the chip moved during the run.
* `correlation_run_order_vs_date`: should be about 0. It confirms the shuffle worked, so that drift cannot line up with dates.
* `noise_reveals_run_position_auc`: can the noise tell whether a circuit ran early or late in the run? 0.5 means no detectable drift.

## 7. How to read the real result

All comparisons use the same events and the same models.

| What you see | What it means |
|---|---|
| D is no better than A | No hardware advantage on this data. Together with Notebook 3, the HSBC effect does not reproduce here. |
| D beats A, **and E matches D** | **It was the noise.** The effect is reproduced by ordinary random numbers, for free. |
| D beats A, **and D beats E** | **Something unexplained.** Hardware adds something our classical copy lacks. Repeat on another day and another chip before believing it. |
| Sentinels scatter much more than shot noise | The chip drifted. Our random order protects us, but it is a warning for anyone who runs time-ordered data. |

## 8. How many events do you need?

Look at the width of the intervals in the rehearsal table: with 600 events (about 300 for testing) the AUC is only known to within roughly ±0.06. So a hardware run of a few hundred events can only detect **large** effects.

Notice also that in the rehearsal even the classical inputs (A) score about 0.5. With only about 300 events to learn from, the weak signal of Notebook 1 cannot be found at all. Expect the same in a small real run: the question will not be "is D a little better than A?" but "does D jump far above everything else?".

That is enough for the question at hand. HSBC report the AUC rising from about 0.63 to between 0.75 and 0.97. An effect of that size would stand far outside ±0.06. A few hundred events cannot, however, detect a small effect such as the 0.02 differences in Notebook 3; that would need thousands of events (the opt-in 180-minute allowance).

## 9. Every limitation of this project in one place

**Data**

1. The fill label is a stand-in: a simple simulated dealer replayed against real trade prices. Real quote requests are private.
2. Only requests that ended in a trade are visible; requests where the customer walked away are not.
3. Daily data, not time-stamped to the microsecond like HSBC's.
4. US corporate bonds, not European.
5. 97 of the 216 listed features (all that are free) plus the side of the request. No ratings, company accounts, share prices, CDS or holdings.
6. Only liquid, seasoned bonds.
7. The signal is weak (classical AUC about 0.55, against about 0.63 for HSBC), so small differences between methods are hard to see.

**Quantum**

8. **No real hardware yet.** Set D is ready to run but has not been run.
9. The simulated noise is a simplified model: random gate errors, readout errors, shot noise and slow drift. Real chips also have crosstalk, leakage and other effects.
10. With 98 inputs instead of 216, each input drives two or three gates. HSBC's "longer" circuit (twice the depth) is not simulated.
11. One circuit, one seed. That is deliberate (no cherry-picking), but other seeds were not explored.

**Statistics**

12. One 68-day window. Markets in other periods may behave differently.
13. Many comparisons are reported; a few "real" differences will appear by chance. We rely on the consistent overall picture, not on any single cell.
14. Our hyper-parameter grids are smaller than HSBC's.

## 10. Ideas for going further

* **Run the hardware.** It is the one thing that can test the claim.
* **Repeat the hardware run on a second day or chip**, to see whether any effect survives.
* **Send one batch in date order** as well as the random-order batch, to demonstrate the clock on real hardware.
* **Try other seeds** for the start state and the feature-to-gate map, and report all of them.
* **A richer fill label**, if anyone can get access to real quote data.

**Back to the start:** [Notebook 0](00_START_HERE.ipynb).